# HeliciteCinetique — `helicite_cinetique`

`Postraitement_fields` · ASN 28 validation form (TRUST-LiMe).


## Introduction

Validation field of the **standalone magnetic (induction) equation** solved by
TRUST-LiMe:

$$\partial_t b + (u \cdot \nabla) b - \nabla \cdot (\eta \nabla b) = 0$$

This notebook runs **one and only one test case**: ``helicite_cinetique`. Per the TRUST validation-form layout
(`Validation/Rapports_automatiques/<Category>/<FormName>/`), the other
`Postraitement_fields` cases live in their own sibling folders.

## What it validates

Kinetic helicity density built from the element gradient of the frozen
velocity. 3D linear velocity $u = (y, z, x)$: $\nabla \times u = -(1,1,1)$
exactly, so $h_k = -(x + y + z)$. The probe sits at the cell centre
$(0.375, 0.375, 0.375)$ where $h_k = -1.125$. (In 2D $h_k \equiv 0$:
out-of-plane vorticity, in-plane velocity.)

## Contract checked

| # | Quantity | Formula |
|---|---|---|
| 1 | kinetic helicity density | $h_k = u \cdot (\nabla \times u)$ |

Computed in the module by the `Champ_Fonc_MHD` post-treatment field
(`src/Champ_Fonc_MHD.{h,cpp}` + `src/MHD_posttraitement_kernel.h`).


In [ ]:
from trustutils import run

run.TRUST_parameters("1.9.8")
run.reset()


## Helpers: read a TRUST probe (`.son`) file

The case is copied by `trustutils` into `run.BUILD_DIRECTORY`; the `.son`
probe files are written next to the dataset in `src/` and copied over in the
run cell below, where `read_son()` expects them.


In [ ]:
import numpy as np
from trustutils.files import SonFile

BUILD = run.BUILD_DIRECTORY

def read_son(case, sonde):
    """Return (time, values) for probe `sonde` of case `case` (dataset name)."""
    path = f"{BUILD}/{case}_{sonde}.son"
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry found in {path}"
    t, v = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(v, dtype=float)

def read_son_all(case, sonde):
    """Like read_son but returns every entry (components of a vector field)."""
    path = f"{BUILD}/{case}_{sonde}.son"
    son = SonFile(path, None)
    return [(e, np.asarray(son.getValues(e)[1], dtype=float)) for e in son.getEntries()]

def final_value(case, sonde):
    """Field value at the probe, at the last recorded time."""
    t, v = read_son(case, sonde)
    # the first line (t=0) is written before the problem is initialised
    return t, v[-1]

print("BUILD =", BUILD)


## Run

Runs on **2 processors** (`nbProcs=2` + `case.partition()`). The dataset's
commented `# BEGIN PARTITION` / `# BEGIN SCATTER` blocks let `trust -partition`
build the parallel `PAR_` variant (upstream `grad_rot.data` pattern).


In [ ]:
import pathlib, shutil

from trustutils import run

NB_PROCS = 2

run.reset()
case = run.addCase(".", "helicite_cinetique.data", nbProcs=NB_PROCS)
case.partition()          # trust -partition helicite_cinetique.data 2 -> PAR_helicite_cinetique.data + DOM_000*.Zones
# --- mesh-convergence scan (shared ladder: L1 = cells x2, L2 = cells x4) ---
# Levels instantiated from src/helicite_cinetique.tpl.data (byte-copy of the baseline
# dataset with $nx/$ny/$nz node-count placeholders); the baseline case above is
# the untouched L0. Each level runs in its own build/scan/<lvl>/ directory
# so that the partitioned DOM_000*.Zones never overwrite another's.
NX0 = NY0 = NZ0 = 5                       # baseline node counts (L0)
LEVELS = {
    "L1": {"nx": 2*5-1, "ny": 2*5-1, "nz": 2*5-1},
    "L2": {"nx": 4*5-3, "ny": 4*5-3, "nz": 4*5-3},
}
scan_cases = {}
for lvl, dic in LEVELS.items():
    c = run.addCaseFromTemplate("helicite_cinetique.tpl.data", f"scan/{lvl}", dic=dic,
                                targetData=f"helicite_cinetique_{lvl}.data",
                                nbProcs=NB_PROCS, excluNR=True)
    c.partition()
    scan_cases[lvl] = c
run.printCases()
run.runCases()

# trustutils runs the case in its build tree; the .son probes are written next
# to the dataset in src/, so copy them where read_son() looks for them.
build = pathlib.Path(run.BUILD_DIRECTORY)
build.mkdir(parents=True, exist_ok=True)
for son in pathlib.Path("src").glob(f"{case.dataFileName_}_*.son"):
    shutil.copy2(son, build / son.name)

## Checks


In [ ]:
t, hk = final_value(case.dataFileName_, "SONDE_HK")

expected = -(0.375 + 0.375 + 0.375)
print(f"final time                : {t[-1]:.6g}")
print(f"helicite_cinetique (probe): {hk:.6g}  (expected {expected:.6g})")
assert abs(hk - expected) < 1e-8, f"helicite_cinetique wrong: {hk}"
print("OK: helicite_cinetique = u . curl(u).")


## Mesh convergence

The shared refinement ladder: `L0` = the baseline mesh above (`5x5x5`),
`L1` = cells x2 (`9x9x9`), `L2` = cells x4 (`17x17x17`).

The field is reconstructed from a linear/uniform `b` at an interior probe,
which is a **discrete identity** (the Green-Gauss reconstruction is exact
for linear fields at interior cells): u . curl(u) for the linear data velocity — position-dependent (3D). The scan asserts the same
machine-precision bound at **every** level
(evaluated at the probe's effective cell-centre position from the .son header) — no order is reported.

In [ ]:
def probe_pos(path):
    """Effective (x, y, z) of a point probe, from the .son header."""
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.startswith("# Temps"):
                toks = line.replace("=", " = ").split()
                return (float(toks[toks.index("x") + 2]), float(toks[toks.index("y") + 2]),
                        float(toks[toks.index("z") + 2]))
    raise RuntimeError(f"no probe position header in {path}")
def read_final_dir(subdir, case_name, name):
    """(time, values) of probe `name` in a build subdir (final value = v[-1])."""
    path = f"{BUILD}/{subdir}/{case_name}_{name}.son"
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry found in {path}"
    t, v = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(v, dtype=float)

def level_error(subdir, case_name):
    _, v = read_final_dir(subdir, case_name, "SONDE_HK")
    xy = probe_pos(f"{BUILD}/{subdir}/{case_name}_SONDE_HK.son")
    expected = -(xy[0] + xy[1] + xy[2])
    return abs(float(v[-1]) - float(expected)), expected

scan = {"L0": ("", case.dataFileName_),
        "L1": (f"scan/L1", scan_cases["L1"].dataFileName_),
        "L2": (f"scan/L2", scan_cases["L2"].dataFileName_)}
scan_nodes = {"L0": (5, 5, 5), "L1": (9, 9, 9), "L2": (17, 17, 17)}
res = {lvl: level_error(sub, cn) for lvl, (sub, cn) in scan.items()}

print(f"level  nodes        abs err        rel")
for lvl in ("L0", "L1", "L2"):
    n0, n1, n2 = scan_nodes[lvl]
    e, expected = res[lvl]
    rel = e
    print(f"{lvl:<6} {n0:>3}x{n1:>3}x{n2:>3} {e:<13.3e} {rel:.1e}")
    assert rel < 1e-8, f"identity broken at {lvl}: {rel:.3e}"
print("OK: the reconstruction identity holds at every mesh level.")

## Computer Performance


In [ ]:
run.tablePerf()
